In [ ]:
import pandas as pd 

# Load combined cleaned pathology data frame
df_path = "D:\DATA\df_cleaned.xlsx"
df_all = pd.read_excel(df_path)
print(df_all.columns)

In [ ]:
# Remove rendundant columns
cols_to_remove = ["mattype tekst", "makrotekst", "mikrotekst", "snomed kode", "kode fritekst","wsi count"]
df_selected = df_all.drop(columns = cols_to_remove)

print(df_selected.head())

In [ ]:
import ast

def convert_string_to_list(df, columns):
    for col in columns:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else x)
    return df

df_selected = convert_string_to_list(df_selected, ["wsi filenames", "T", "M", "Other"])

In [ ]:
from snomed_hierarchy import SNOMEDCodes, SNOMEDHierarchy

# Load SNOMED codes
snomed_path = "D:/DATA/patoSnoMed_2025-04.xlsx"
xls_snomed = pd.read_excel(snomed_path)
df_snomed = pd.DataFrame(xls_snomed, columns=['SKSkode', 'DatoFra', 'DatoÆndring', 'DatoTil', 'Kodetekst', 'Fuldtekst'])
snomed = SNOMEDCodes(df_snomed)

In [ ]:
# Create morphology hierarchy

# Get df of M codes
m_codes = snomed.get_codes_by_letter('M')

all_m_codes = df_selected["M"].explode().unique()
m_filtered = m_codes[m_codes["SKSkode"].isin(all_m_codes)]

# Build hierarchy
m_hierarchy = SNOMEDHierarchy(m_filtered, main_len=2)

print("Number of available m codes: ", len(m_codes))
print("Number of m codes present in data set: ", len(m_filtered))

In [ ]:
# Create topography hierarchy

# Get df of T codes
t_codes = snomed.get_codes_by_letter('T')

all_t_codes = df_selected["T"].explode().unique()
t_filtered = t_codes[t_codes["SKSkode"].isin(all_t_codes)]

# Build hierarchy
t_hierarchy = SNOMEDHierarchy(t_filtered, main_len=3)

print("Number of available t codes: ", len(t_codes))
print("Number of t codes present in data set: ", len(t_filtered))

In [ ]:
t_hierarchy.print_all_regions(edited=False)

In [ ]:
# Manually edit T hierarchy

# Update region name
t_hierarchy.update_region('T00', 'Uspecifik topografi')

# Merge regions T01, T02 and T03 into one main region T01+
t_hierarchy.merge_main_regions('T01+', ['T01', 'T02', 'T03'], new_name="Hud inkl. subcutis")

# Split T1X into T1X0 (Bløddelsvæv) and T1X+ (Knogle- og bruskvæv)
t_hierarchy.split_main_region('T1X', {'T1X0': ['T1X0'], 'T1X+': ['T1X5','T1X7']})

# Other updates
t_hierarchy.update_region('T04', 'Mammae')
t_hierarchy.merge_main_regions('T08+', ['T08', 'T09'], new_name="Lymfeknude og lymfekar")
t_hierarchy.merge_main_regions('T10+', ['T10', 'T11', 'T1X+'], new_name="Knogle")
t_hierarchy.update_region('T24', 'Larynx')
t_hierarchy.split_main_region('T2Y', {'T2Y4': ['T2Y4'], 'T2Y6': ['T2Y6']})
t_hierarchy.merge_main_regions('T26+', ['T26', 'T2Y4'], new_name="Bronchus")
t_hierarchy.merge_main_regions('T29+', ['T29', 'T2Y6'], new_name="Pleura")
t_hierarchy.merge_main_regions('T40+', ['T40', 'T45', 'T46', 'T48'], new_name="Blodkar")
t_hierarchy.update_region('T67', 'Colon')
t_hierarchy.update_region('T68', 'Rectum')
t_hierarchy.merge_main_regions('T71+', ['T71', 'T72'], new_name="Nyrer")
t_hierarchy.merge_main_regions('T74', ['T74', 'T7X'], new_name="Urinblære")
t_hierarchy.update_region('T79', 'Øvrige hanlige kønsorganer')
t_hierarchy.update_region('T93', 'Binyre')
t_hierarchy.update_region('TX2', 'Hjerne')
t_hierarchy.update_region('TXX', 'Øje')
t_hierarchy.merge_main_regions('T83+', ['T83','T8X'], new_name="Cervix Uteri")
t_hierarchy.merge_main_regions('T88+', ['T88','T89'])
# CONTINUE FROM T8X
# CONSIDERATIONS: broader groups, see below
"""
1. General / Undefined
T00 Topografi ukendt

2. Integumentary System (Hud og subcutis)
Hud (T01)
Hud efter region (T02)
Subcutis (T03)
Mamma (T04)

3. Hematopoietic & Lymphoid
Knoglemarv (T06)
Milt (T07)
Lymfeknuder (T08)
Lymfekar (T09)

4. Skelet & Bevægelsesapparat
Knogle (T10–T11)
Led (T12)
Muskler, sener og støttevæv
Skeletmuskulatur (T13)
Bursa (T16)
Sene (T17)
Ligament og fascie (T18)
Bløddelstypologi (T1X)

5. Luftveje
Næse & bihuler (T21–T22)
Pharynx / svælg (T23, T60–T63)
Tonsiller, adenoid og tilhørende væv (T61–T613)
Larynx & stemmebånd (T24)
Trachea & bronkier (T25–T26)
Lunger & pleura (T28–T29)
Cytologi luftveje (T2Y)

6. Hjerte & Kar
Endocardium (T34)
Blodkar (T40-T45-T46)
Vener (T48)

7. Fordøjelsessystemet
Mundhule (T51–T55)
Lever, galde, pancreas (T56–T59)
Mave-tarmkanalen (T63-T64-T65–T69)

8. Urinveje
Nyrer (T71)
Nyrepelvis (T72)
Ureter (T73)
Blære (T74)
Urethra (T75)
Cytologi urinveje (T7X)

9. Mandlige genitalia
Penis (T76)
Prostata & vesicula seminalis (T77)
Testis, epididymis, ductus deferens, funiculus spermaticus, scrotum (T78–T79)

10. Kvindelige genitalia
Vulva, labia, clitoris, Bartholin (T80)
Vagina (T81)
Uterus & cervix (T82–T83)
Endometrium, myometrium (T84–T85)
Tuba uterina & ovarier (T86–T87)
Placenta, fosterhinder, navlestreng (T88)
Foster (T89)
Cytologi cervix (T8X)

11. Endokrine kirtler
Binyre (T93)
Thyroidea & parathyroidea (T96–T97)
Thymus (T98)

12. Nervesystem & Sanseorganer
Hjerne (temporallap) (TX2)
Nerver (TX9)
Øjenlåg / conjunctiva (TXX)
Øre (ydre, mellemøre, øregang) (TXY)

13. Regionbaserede strukturer (TY-serie)
Hoved & hals (TY0)
Truncus (ryg, thorax, abdomen, pelvis, inguen) (TY1–TY7)
Ekstremiteter (over- og underekstremitet) (TY8–TY9)
"""

In [ ]:
t_hierarchy.list_main_regions(edited = True)

In [ ]:
m_hierarchy.print_all_regions(edited=False)

In [ ]:
# Manually edit M hierarchy
m_hierarchy.update_region('M0', 'Unspecific morphology')
m_hierarchy.update_region('M1', 'Traumatic changes')
m_hierarchy.update_region('M2', 'Congenital malformations, pregnancy products')
m_hierarchy.update_region('M3', 'Mechanical changes')
m_hierarchy.update_region('M4', 'Inflammation and fibrosis')
m_hierarchy.update_region('M5', 'Degeneration, necrosis, deposition, dystrophy, atrophy') 
m_hierarchy.update_region('M6', 'Cellular changes') 
m_hierarchy.update_region('M7', 'Growth and maturation changes') 
m_hierarchy.merge_main_regions('M8-9', ['M8', 'M9'], new_name="Neoplasms")
m_hierarchy.update_region('MÆ', 'Description in text')

In [ ]:
m_hierarchy.list_main_regions(edited = True)

In [ ]:
def map_codes_to_category(code_list, hierarchy):
    categories = []
    for code in code_list:
        region_name = hierarchy.code_to_main_region_name(code)
        categories.append(region_name)
    return list(set(categories))

In [ ]:
df_selected["T category"] = df_selected["T"].apply(lambda x: map_codes_to_category(x, t_hierarchy))
df_selected["M category"] = df_selected["M"].apply(lambda x: map_codes_to_category(x, m_hierarchy))


In [ ]:
print(df_selected.head())
